# CamAiSearch en Google Colab

Este notebook instala dependencias, levanta la API y permite analizar un video desde `videos/`.

In [ ]:
import os
import subprocess
from pathlib import Path

repo_dir = Path("/content/CamAiSearch")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/CarlosGTI001/CamAiSearch.git", str(repo_dir)], check=True)

os.chdir(repo_dir)
print("Working directory:", Path.cwd())

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)

In [ ]:
from pathlib import Path
from google.colab import files

Path("videos").mkdir(parents=True, exist_ok=True)
uploaded = files.upload()

for filename, content in uploaded.items():
    output = Path("videos") / filename
    output.write_bytes(content)
    print("Saved:", output)

In [ ]:
import subprocess
import sys
import time

cmd = [
    sys.executable,
    "main.py",
    "--config",
    "config/config.colab.json",
    "--host",
    "0.0.0.0",
    "--port",
    "8000",
]

api_proc = subprocess.Popen(cmd)
time.sleep(8)
print("API PID:", api_proc.pid)

In [ ]:
import json
import urllib.request

with urllib.request.urlopen("http://127.0.0.1:8000/events") as response:
    payload = json.loads(response.read().decode("utf-8"))

print("Events loaded:", len(payload))

In [ ]:
import json
import urllib.request

body = json.dumps({
    "source": "videos/tu_video.mp4",
    "camera_id": "colab_demo",
    "max_seconds": 60,
    "save_clips": True
}).encode("utf-8")

request = urllib.request.Request(
    "http://127.0.0.1:8000/analyze",
    data=body,
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(request) as response:
    print(response.read().decode("utf-8"))

In [ ]:
if "api_proc" in globals() and api_proc.poll() is None:
    api_proc.terminate()
    api_proc.wait(timeout=10)
    print("API stopped.")
else:
    print("API was not running.")